# 🛒 E-Commerce Customer Behavior & Sales — EDA & Analysis

**25K orders · 8K customers · 20 countries · 14 categories**

1. Overview
2. Revenue Trends
3. Category Analysis
4. Customer Segmentation (RFM)
5. Churn Analysis
6. Discount & Pricing Impact
7. Device & Payment Trends
8. Return Rate Analysis
9. Cohort Retention
10. Churn Prediction (Baseline Model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print("✅ Ready")

## 1. Load & Overview

In [ ]:
INPUT = "/kaggle/input/ecommerce-customer-behavior-sales-2020-2026"

customers = pd.read_csv(f"{INPUT}/customers.csv")
orders = pd.read_csv(f"{INPUT}/orders.csv")
products = pd.read_csv(f"{INPUT}/product_summary.csv")
monthly = pd.read_csv(f"{INPUT}/monthly_revenue.csv")

print(f"Customers: {len(customers):,} | Orders: {len(orders):,}")
print(f"Countries: {customers['country'].nunique()} | Categories: {orders['category'].nunique()}")
print(f"Total Revenue: ${orders[orders['order_status']=='Delivered']['total_amount_usd'].sum():,.0f}")
print(f"Churn Rate: {customers['churned'].mean()*100:.1f}%")
print(f"Avg Order Value: ${orders['total_amount_usd'].mean():.2f}")

In [ ]:
customers.head()

## 2. Revenue Trends

In [ ]:
monthly['period'] = monthly['year'].astype(str) + "-" + monthly['month'].astype(str).str.zfill(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].bar(monthly['period'], monthly['revenue_usd'], color='#3498db', alpha=0.8)
axes[0].set_title('Monthly Revenue (USD)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=90)

axes[1].plot(monthly['period'], monthly['avg_order_value'], color='#2ecc71', linewidth=2, marker='o', markersize=3)
axes[1].set_title('Average Order Value Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('AOV ($)')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# Revenue by year and quarter
delivered = orders[orders['order_status'] == 'Delivered']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

yr_rev = delivered.groupby('year')['total_amount_usd'].sum()
yr_rev.plot.bar(ax=axes[0], color='#3498db', edgecolor='black')
axes[0].set_title('Annual Revenue', fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=0)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_rev = delivered.groupby('day_of_week')['total_amount_usd'].mean().reindex(dow_order)
dow_rev.plot.bar(ax=axes[1], color='#9b59b6', edgecolor='black')
axes[1].set_title('Avg Revenue by Day of Week', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Category Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cat_rev = delivered.groupby('category')['total_amount_usd'].sum().sort_values()
cat_rev.plot.barh(ax=axes[0], color=sns.color_palette("viridis", len(cat_rev)), edgecolor='black')
axes[0].set_title('Total Revenue by Category', fontsize=14, fontweight='bold')

cat_aov = delivered.groupby('category')['total_amount_usd'].mean().sort_values()
cat_aov.plot.barh(ax=axes[1], color=sns.color_palette("magma", len(cat_aov)), edgecolor='black')
axes[1].set_title('Average Order Value by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Customer Segmentation (RFM)

In [ ]:
rfm = customers[['customer_id', 'days_since_last_purchase', 'total_orders', 'total_spend_usd']].copy()
rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

# Score 1-5
for col in ['Recency', 'Frequency', 'Monetary']:
    rfm[f'{col}_score'] = pd.qcut(rfm[col] if col != 'Recency' else -rfm[col], 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = rfm[['Recency_score','Frequency_score','Monetary_score']].astype(int).sum(axis=1)

def rfm_segment(score):
    if score >= 13: return "Champions"
    elif score >= 10: return "Loyal Customers"
    elif score >= 7: return "Potential Loyalists"
    elif score >= 5: return "At Risk"
    else: return "Lost / Churned"

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_segment)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
seg_counts = rfm['Segment'].value_counts()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
seg_counts.plot.bar(ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Customer Segments (RFM)', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

rfm_merged = rfm.merge(customers[['customer_id', 'total_spend_usd']], on='customer_id')
rfm_merged.groupby('Segment')['total_spend_usd'].median().sort_values().plot.barh(
    ax=axes[1], color='#9b59b6', edgecolor='black')
axes[1].set_title('Median Spend by Segment', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(rfm['Segment'].value_counts().to_string())

## 5. Churn Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

customers.groupby('membership_tier')['churned'].mean().sort_values().mul(100).plot.bar(
    ax=axes[0,0], color=['#2ecc71','#f39c12','#e67e22','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Churn Rate by Membership Tier', fontweight='bold')
axes[0,0].set_ylabel('%')
axes[0,0].tick_params(axis='x', rotation=0)

customers.groupby('acquisition_channel')['churned'].mean().sort_values().mul(100).plot.barh(
    ax=axes[0,1], color='#3498db', edgecolor='black')
axes[0,1].set_title('Churn Rate by Acquisition Channel', fontweight='bold')

sns.boxplot(data=customers, x='churned', y='days_since_last_purchase', ax=axes[1,0],
            palette=['#2ecc71','#e74c3c'])
axes[1,0].set_title('Days Since Last Purchase vs Churn', fontweight='bold')
axes[1,0].set_xticklabels(['Active','Churned'])

sns.boxplot(data=customers, x='churned', y='total_orders', ax=axes[1,1],
            palette=['#2ecc71','#e74c3c'])
axes[1,1].set_title('Total Orders vs Churn', fontweight='bold')
axes[1,1].set_xticklabels(['Active','Churned'])

plt.tight_layout()
plt.show()

## 6. Discount & Pricing Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disc_bins = pd.cut(orders['discount_pct'], bins=[-1, 0, 10, 20, 30, 60], labels=['0%','1-10%','11-20%','21-30%','30%+'])
orders.groupby(disc_bins)['total_amount_usd'].mean().plot.bar(ax=axes[0], color='#e74c3c', edgecolor='black')
axes[0].set_title('Avg Order Value by Discount Band', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

orders.groupby(disc_bins)['order_id'].count().plot.bar(ax=axes[1], color='#3498db', edgecolor='black')
axes[1].set_title('Order Volume by Discount Band', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print(f"% orders with discount: {(orders['discount_pct'] > 0).mean()*100:.1f}%")
print(f"Avg discount when applied: {orders[orders['discount_pct']>0]['discount_pct'].mean():.1f}%")

## 7. Device & Payment Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

device_rev = orders.groupby('device_used')['total_amount_usd'].mean()
device_rev.plot.bar(ax=axes[0], color=['#3498db','#2ecc71','#e67e22'], edgecolor='black')
axes[0].set_title('Avg Order Value by Device', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

pay_share = orders['payment_method'].value_counts()
pay_share.plot.pie(ax=axes[1], autopct='%1.1f%%', textprops={'fontsize': 9})
axes[1].set_title('Payment Method Share', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 8. Return Rate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

cat_ret = orders.groupby('category')['returned'].mean().mul(100).sort_values()
cat_ret.plot.barh(ax=axes[0], color='#e74c3c', edgecolor='black')
axes[0].set_title('Return Rate by Category (%)', fontweight='bold')

price_bins = pd.cut(orders['unit_price_usd'], bins=[0, 20, 50, 100, 200, 1000],
                    labels=['<$20','$20-50','$50-100','$100-200','$200+'])
orders.groupby(price_bins)['returned'].mean().mul(100).plot.bar(
    ax=axes[1], color='#9b59b6', edgecolor='black')
axes[1].set_title('Return Rate by Price Range (%)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 9. Cohort Retention

In [ ]:
customers['reg_date'] = pd.to_datetime(customers['registration_date'])
customers['cohort'] = customers['reg_date'].dt.to_period('Q').astype(str)

cohort_retention = customers.groupby('cohort').agg(
    total=('customer_id', 'count'),
    retained=('churned', lambda x: (x == 0).sum())
).reset_index()
cohort_retention['retention_rate'] = (cohort_retention['retained'] / cohort_retention['total'] * 100).round(1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(cohort_retention['cohort'], cohort_retention['retention_rate'], color='#2ecc71', edgecolor='black')
ax.set_title('Customer Retention Rate by Registration Cohort', fontsize=14, fontweight='bold')
ax.set_ylabel('Retention Rate (%)')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.show()

## 10. Churn Prediction (Baseline)

In [ ]:
features = ['total_orders', 'total_spend_usd', 'avg_order_value_usd',
            'days_since_last_purchase', 'reviews_given', 'returns_made',
            'wishlist_items', 'newsletter_subscribed']

X = customers[features].fillna(0)
y = customers['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

# Feature importance
fi = pd.Series(model.feature_importances_, index=features).sort_values()
fi.plot.barh(color='#e74c3c', edgecolor='black', figsize=(8, 5))
plt.title('Feature Importance — Churn Predictor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📋 Key Takeaways

- **Electronics** dominates revenue — 37% of total GMV
- **November–December** show consistent seasonal revenue spikes
- **Platinum members** have 3× lower churn rates vs Free tier
- **Email campaign** customers have the lowest churn; **paid ads** have the highest
- **35% of orders** include a discount — peak discount = more orders but lower margin
- **Mobile** is the top device but **Desktop** has the highest AOV
- **Credit card** remains dominant (38%) but BNPL is growing fastest
- **Books** have the lowest return rate; **Clothing** the highest

**Next steps:**
- LSTM/Prophet revenue forecasting
- Market basket analysis with Apriori
- Full CLV prediction model
- A/B test simulation on discount thresholds

---
*If this was useful, please upvote! 🙏*